In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [34]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.structured-OCR.read-vehicle-title",
        description="Extract relevent information from vehicle title",
        worker_release="qwen3-instruct",
        text_prompt="""
        You are given an image of a vehicle title document (certificate of title).

        Your task is to extract structured data that is clearly visible in the image.

        Return ONLY valid JSON.
        Do not include explanation.
        Do not include markdown.
        Do not include commentary.

        ----------------------------------------
        INSTRUCTIONS:

        1. Extract only text that is clearly readable in the image.
        2. Do NOT guess or infer missing values.
        3. If a field is not visible or unreadable, return null.
        4. Preserve capitalization exactly as shown.
        5. Preserve spacing inside fields where applicable (names, VIN, addresses).
        6. Do NOT reformat dates unless the format is explicitly clear in the document.
        7. Do NOT fabricate data.
        8. If the image is not a vehicle title document, return:
        {
          "document_type": "unknown"
        }

        ----------------------------------------
        RETURN THIS EXACT JSON STRUCTURE:

        {
          "document_type": "vehicle_title",
          "jurisdiction": {
            "issuing_state_or_country": null,
            "issuing_agency": null
          },
          "title_number": null,
          "issue_date": null,
          "print_date": null,
          "vehicle": {
            "vin": null,
            "year": null,
            "make": null,
            "model": null,
            "trim": null,
            "body_style": null,
            "color": null,
            "fuel_type": null,
            "odometer": {
              "reading": null,
              "unit": null,
              "as_of_date": null,
              "status": null
            }
          },
          "owner": {
            "name": null,
            "address": {
              "street": null,
              "city": null,
              "state_or_region": null,
              "postal_code": null
            }
          },
          "co_owner": {
            "name": null,
            "address": {
              "street": null,
              "city": null,
              "state_or_region": null,
              "postal_code": null
            }
          },
          "lienholder": {
            "name": null,
            "address": {
              "street": null,
              "city": null,
              "state_or_region": null,
              "postal_code": null
            },
            "lien_date": null
          },
          "branding_and_status": {
            "title_brand": null,
            "salvage_brand": null,
            "rebuilt_brand": null,
            "flood_brand": null,
            "lemon_brand": null,
            "export_only": null,
            "junk_brand": null,
            "odometer_brand": null
          },
          "fees_and_taxes": {
            "total_fees": null,
            "taxes": null,
            "amount_paid": null
          },
          "signatures": {
            "owner_signed": null,
            "co_owner_signed": null,
            "authorized_agent_signed": null
          },
          "raw_text_notes": null
        }

        ----------------------------------------
        FIELD NOTES / NORMALIZATION:

        - vin: must be 17 characters if fully visible; otherwise return the partial visible string or null if unreadable.
        - odometer.status: one of ["actual", "not_actual", "exempt", null] if the wording is clearly present; otherwise null.
        - branding_and_status fields: return the exact visible words (e.g., "SALVAGE", "REBUILT", "FLOOD") if present; otherwise null.
        - signatures: set to true only if you can clearly see a signature mark; set to false only if the signature line is clearly present and blank; otherwise null.

        ----------------------------------------
        AMBIGUITY RULE:

        If multiple candidates exist for a value, choose the one most clearly labeled and most prominent as an official field value.

        Return only the JSON object.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=700,
            image_size=512
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.structured-OCR.read-vehicle-title:latest"
   )
])


with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("sample_img.png")
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")

### Evaluation Flow

In [ ]:
from pathlib import Path

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.structured-OCR.read-vehicle-title:latest"
    )
])

all_results = {}

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)
    directory_path = Path("/content/sample_data")
    for item in directory_path.iterdir():
        job = endpoint.upload(str(item))
        file_results = []

        while result := job.predict():
            file_results.append(result)

        all_results[item.name] = file_results

output_path = Path("/content/sample_data/output.json")
with open(output_path, "w") as f:
  json.dump(all_results, f, indent=2)

print("Done")